# 🚀 GenAI Restoration Studio - Google Colab T4 Bootstrap & Setup

This bootstrap notebook initializes the Google Colab T4 GPU environment, connects to Google Drive for persistent datasets/checkpoints/MLflow logs, pulls the latest project code, and verifies the data pipelines and PyTorch dataset modules.

### Step 1: Mount Google Drive & Configure Persistent Paths

In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define persistent paths in Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/GenAI-A1'
RAW_OXFORD_DIR = os.path.join(DRIVE_ROOT, 'raw', 'OxfordPet')
RAW_FS2K_DIR = os.path.join(DRIVE_ROOT, 'raw', 'FS2K')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
MLRUNS_DIR = os.path.join(DRIVE_ROOT, 'mlruns')
MANIFESTS_DIR = os.path.join(DRIVE_ROOT, 'manifests')
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')

for p in [DRIVE_ROOT, RAW_OXFORD_DIR, RAW_FS2K_DIR, CHECKPOINTS_DIR, MLRUNS_DIR, MANIFESTS_DIR, EXPORTS_DIR]:
    os.makedirs(p, exist_ok=True)

print("✅ Google Drive mounted and directories initialized:")
print(f"  Persistent Root: {DRIVE_ROOT}")
print(f"  Checkpoints:     {CHECKPOINTS_DIR}")
print(f"  MLflow Logs:     {MLRUNS_DIR}")

Mounted at /content/drive
✅ Google Drive mounted and directories initialized:
  Persistent Root: /content/drive/MyDrive/GenAI-A1
  Checkpoints:     /content/drive/MyDrive/GenAI-A1/checkpoints
  MLflow Logs:     /content/drive/MyDrive/GenAI-A1/mlruns


### Step 2: Clone or Update Project Repository

In [4]:
import sys

REPO_URL = 'https://github.com/UsmanBari/genai-restoration-studio.git'
REPO_DIR = '/content/genai-restoration-studio'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("✅ Working directory set to repo:", os.getcwd())

/content/genai-restoration-studio
From https://github.com/UsmanBari/genai-restoration-studio
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/genai-restoration-studio
✅ Working directory set to repo: /content/genai-restoration-studio


### Step 3: Install Training Dependencies & Check CUDA GPU

In [5]:
!pip install -q -r requirements-colab.txt

import torch
import torchvision
import optuna
import mlflow
import onnx
import onnxruntime

print("✅ PyTorch version:", torch.__version__)
print("✅ CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU Device:", torch.cuda.get_device_name(0))
    print("✅ Memory Allocated:", round(torch.cuda.memory_allocated(0)/(1024**3), 2), "GB")
else:
    print("⚠️ Warning: Running on CPU. Switch Colab runtime to T4 GPU (Runtime > Change runtime type).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

### Step 4: Prepare Oxford-IIIT Pet Dataset (Run Once)

In [6]:
from scripts.prepare_oxford_pet import prepare_oxford_pet

print("Preparing Oxford-IIIT Pet Dataset on Google Drive...")
prepare_oxford_pet(
    output_dir=RAW_OXFORD_DIR,
    manifest_dir=MANIFESTS_DIR,
    target_size=(128, 128),
    split_seed=42
)
print("✅ Oxford-IIIT Pet Dataset ready!")

Preparing Oxford-IIIT Pet Dataset on Google Drive...
Downloaded /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/images.tar.gz.
Downloaded /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/annotations.tar.gz.
Extracting /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/images.tar.gz to /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/raw_extracted...
Extraction complete.
Extracting /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/annotations.tar.gz to /content/drive/MyDrive/GenAI-A1/raw/OxfordPet/raw_extracted...
Extraction complete.
Resizing 7349 images to (128, 128) RGB...


100%|██████████| 7349/7349 [02:19<00:00, 52.81it/s]


Generating deterministic train/val/test manifests...
Manifests successfully created at:
  /content/drive/MyDrive/GenAI-A1/manifests/oxford_train_manifest.json
  /content/drive/MyDrive/GenAI-A1/manifests/oxford_val_manifest.json
  /content/drive/MyDrive/GenAI-A1/manifests/oxford_test_manifest.json
✅ Oxford-IIIT Pet Dataset ready!


### Step 5: Unpack & Prepare FS2K Dataset (Run Once)

In [7]:
from scripts.prepare_fs2k import prepare_fs2k

print("Unpacking and verifying FS2K Dataset on Google Drive...")
prepare_fs2k(
    fs2k_dir=RAW_FS2K_DIR,
    manifest_dir=MANIFESTS_DIR,
    split_seed=42
)
print("✅ FS2K Dataset ready!")

Unpacking and verifying FS2K Dataset on Google Drive...
Extracting FS2K zip: /content/drive/MyDrive/GenAI-A1/raw/FS2K/FS2K.zip...
Extraction finished.
Found FS2K root at: /content/drive/MyDrive/GenAI-A1/raw/FS2K/FS2K
Official FS2K counts: 1058 train pairs + 1046 test pairs = 2104 total pairs.
Generating stratified train / validation / test manifests...
Split counts after 15% stratified carve-out:
  Train: 898
  Validation (15% stratified): 160
  Test: 1046
  Total: 2104
Style distribution in Train: {0: 303, 1: 297, 2: 298}
Style distribution in Val:   {0: 54, 1: 53, 2: 53}
Style distribution in Test:  {1: 381, 0: 619, 2: 46}
Manifests successfully created at:
  /content/drive/MyDrive/GenAI-A1/manifests/fs2k_train_manifest.json
  /content/drive/MyDrive/GenAI-A1/manifests/fs2k_val_manifest.json
  /content/drive/MyDrive/GenAI-A1/manifests/fs2k_test_manifest.json
✅ FS2K Dataset ready!


### Step 6: Verify PyTorch DataLoaders & Corruption Pipeline

In [9]:
import matplotlib.pyplot as plt
from data.oxford_pet import get_oxford_dataloaders, OxfordPetDataset
from data.fs2k import get_fs2k_dataloaders, FS2KDataset
from data.corruptions import CORRUPTION_NAMES

oxford_images_dir = os.path.join(RAW_OXFORD_DIR, 'images_128x128')
train_loader, val_loader, test_loader = get_oxford_dataloaders(
    manifest_dir=MANIFESTS_DIR,
    images_dir=oxford_images_dir,
    batch_size=8,
    num_workers=2
)

# Fetch a training batch with runtime corruptions
batch = next(iter(train_loader))
corrupted_imgs = batch['corrupted']
clean_imgs = batch['clean']
labels = batch['label']

print(f"Batch loaded successfully! Corrupted tensor shape: {corrupted_imgs.shape}")
print(f"Corruption labels in batch: {[CORRUPTION_NAMES[l.item()] for l in labels]}")
print("✅ PyTorch DataLoaders and programmatic corruptions verified!")

Batch loaded successfully! Corrupted tensor shape: torch.Size([8, 3, 128, 128])
Corruption labels in batch: ['rectangular_occlusion', 'clean', 'salt_and_pepper', 'clean', 'clean', 'salt_and_pepper', 'salt_and_pepper', 'salt_and_pepper']
✅ PyTorch DataLoaders and programmatic corruptions verified!


### Step 7: Configure MLflow Persistent Tracking

In [10]:
import mlflow

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

db_path = os.path.join(MLRUNS_DIR, "mlflow.db").replace('\\', '/')
mlflow_uri = f"sqlite:///{db_path}"
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("Task1-Universal-Restoration")

with mlflow.start_run(run_name="colab_bootstrap_verification") as run:
    mlflow.log_param("device", "cuda" if torch.cuda.is_available() else "cpu")
    mlflow.log_metric("bootstrap_status", 1.0)

print(f"✅ MLflow SQLite tracking active at {mlflow_uri}")
print("🎉 Environment fully verified and ready for model training!")

2026/09/25 08:57:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/25 08:57:43 INFO mlflow.store.db.utils: Updating database tables
2026/09/25 08:57:48 INFO mlflow.tracking.fluent: Experiment with name 'Task1-Universal-Restoration' does not exist. Creating a new experiment.


✅ MLflow SQLite tracking active at sqlite:////content/drive/MyDrive/GenAI-A1/mlruns/mlflow.db
🎉 Environment fully verified and ready for model training!
